In [1]:
import sys
!{sys.executable} -m pip install --upgrade --force-reinstall \
    azureml-widgets \
    azureml-sdk[automl,notebooks,train] \
    azureml-train-automl-runtime \
    pandas \
    scikit-learn


  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
  Using cached setuptools-82.0.1-py3-none-any.whl.metadata (6.5 kB)
INFO: pip is looking at multiple versions of ipykernel to determine which version is compatible with other requirements. This could take a while.
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 14.1/14.1 MB 73.6 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.6/1.6 MB 14.4 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 26.2/26.2 MB 109.1 MB/s  0:00:00m0:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 39.9/39.9 MB 43.6 MB/s  0:00:00m0:00:0100:01m
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.1/12.1 MB 143.3 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 26.5/26.5 MB 45.7 MB/s  0:00:00 eta 0:00:01m
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.8/1.8 MB 43.5 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.4/1.4 MB 60.7 MB/s  0:00:00
   ━━━━━━━━━

In [2]:
import azureml.core
from azureml.core import Workspace, Experiment, Environment, ScriptRunConfig
from azureml.widgets import RunDetails

# These are the absolute direct paths
import azureml.train.hyperdrive as hd
from azureml.train.hyperdrive.runconfig import HyperDriveConfig
from azureml.train.hyperdrive.sampling import RandomParameterSampling
from azureml.train.hyperdrive.policy import BanditPolicy
from azureml.train.hyperdrive.run import PrimaryMetricGoal
from azureml.train.hyperdrive.parameter_expressions import choice, uniform

print(f" Success! Using SDK Version: {azureml.core.VERSION}")


 Success! Using SDK Version: 1.61.0


In [4]:
from azureml.core import Workspace, Experiment
from azureml.core.compute import ComputeTarget

ws = Workspace.from_config()

exp = Experiment(
    workspace=ws,
    name="heart-failure-hyperdrive"
)

print('Workspace name: ' + ws.name,
      'Azure region: ' + ws.location,
      'Subscription id: ' + ws.subscription_id,
      'Resource group: ' + ws.resource_group,
      sep='\n')


cluster_name = "Compute-cluster1"

cpu_cluster = ComputeTarget(workspace=ws, name=cluster_name)

print("Compute cluster ready")


Workspace name: quick-starts-ws-299204
Azure region: westeurope
Subscription id: 1b944a9b-fdae-4f97-aeb1-b7eea0beac53
Resource group: aml-quickstarts-299204
Compute cluster ready


In [5]:
from azureml.core import Environment, ScriptRunConfig
from azureml.widgets import RunDetails
from azureml.train.hyperdrive.runconfig import HyperDriveConfig
from azureml.train.hyperdrive.run import PrimaryMetricGoal
from azureml.train.hyperdrive.policy import BanditPolicy
from azureml.train.hyperdrive.sampling import RandomParameterSampling
from azureml.train.hyperdrive.parameter_expressions import choice, uniform

# Parameter sampling
ps = RandomParameterSampling({
    "--C": uniform(0.01, 1.0),
    "--max_iter": choice(100, 200, 300)
})

# Early termination policy
policy = BanditPolicy(
    evaluation_interval=2,
    slack_factor=0.1
)

# Environment
sklearn_env = Environment.from_conda_specification(
    name='sklearn-env',
    file_path='conda_dependencies.yml'
)

# Script configuration
src = ScriptRunConfig(
    source_directory='.',
    script='train.py',
    compute_target=cpu_cluster,
    environment=sklearn_env
)

# HyperDrive configuration
hyperdrive_config = HyperDriveConfig(
    run_config=src,
    hyperparameter_sampling=ps,
    policy=policy,
    primary_metric_name='AUC',
    primary_metric_goal=PrimaryMetricGoal.MAXIMIZE,
    max_total_runs=20,
    max_concurrent_runs=4
)

print("HyperDrive configuration ready")


HyperDrive configuration ready


In [7]:
# Submit experiment
hyperdrive_run = exp.submit(hyperdrive_config)

# Wait for completion
hyperdrive_run.wait_for_completion(show_output=True)

print("HyperDrive run completed")


RunId: HD_22697676-036d-4d48-adfb-639a21f9b01b
Web View: https://ml.azure.com/runs/HD_22697676-036d-4d48-adfb-639a21f9b01b?wsid=/subscriptions/1b944a9b-fdae-4f97-aeb1-b7eea0beac53/resourcegroups/aml-quickstarts-299204/workspaces/quick-starts-ws-299204&tid=660b3398-b80e-49d2-bc5b-ac1dc93b5254

Streaming azureml-logs/hyperdrive.txt

[2026-05-08T08:48:30.3642597Z][GENERATOR][DEBUG]Sampled 4 jobs from search space 
[2026-05-08T08:48:30.6342325Z][SCHEDULER][INFO]Scheduling job, id='HD_22697676-036d-4d48-adfb-639a21f9b01b_0' 
[2026-05-08T08:48:30.8461560Z][SCHEDULER][INFO]Scheduling job, id='HD_22697676-036d-4d48-adfb-639a21f9b01b_3' 
[2026-05-08T08:48:30.9921361Z][SCHEDULER][INFO]Scheduling job, id='HD_22697676-036d-4d48-adfb-639a21f9b01b_1' 
[2026-05-08T08:48:30.9932836Z][SCHEDULER][INFO]Scheduling job, id='HD_22697676-036d-4d48-adfb-639a21f9b01b_2' 
[2026-05-08T08:48:31.0712730Z][SCHEDULER][INFO]Successfully scheduled a job. Id='HD_22697676-036d-4d48-adfb-639a21f9b01b_0' 
[2026-05-08T08:4

In [8]:
%pip install joblib

Note: you may need to restart the kernel to use updated packages.


In [9]:
import joblib
# Get your best run and save the model from that run.
best_run = hyperdrive_run.get_best_run_by_primary_metric()

# Metrics
best_run_metrics = best_run.get_metrics()

print('Best AUC:', best_run_metrics.get('AUC'))
print('Accuracy:', best_run_metrics.get('Accuracy'))

# Register model
model = best_run.register_model(
    model_name='hyperdrive-best-model',
    model_path='outputs/model.joblib'
)

print("Model registered successfully!")
    

Best AUC: 0.8639281129653402
Accuracy: 0.8166666666666667
Model registered successfully!


In [10]:
# =========================
# RESOURCE CLEANUP
# =========================

cpu_cluster.delete()
print("Compute cluster deleted!")


Compute cluster deleted!
